# RAILGUN+ — Step 2: Evaluate the v2 Model

Same eval methodology as `02_evaluate.ipynb` and `03_step1_variants.ipynb`, but with:
- The v2 model (9 input channels, feature-engineered)
- **Adaptive-density test sets** that actually produce 50 instances at EVERY agent count (no more 28/12 shrinkage at 96/128). Density is lowered automatically for high-agent counts.

Direct comparison goal: does the v2 model's `corrected` line beat `pibt_only`?

## 1. Setup

In [ ]:
import os, sys, pickle
!git clone -q https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
!pip install -q pogema
sys.path.insert(0,'/kaggle/working/railgun-plus/src')
import torch

CKPT='/kaggle/input/your-checkpoints-v2/best.pt'
LACAM_BIN='/kaggle/input/lacam-binary/main'
if LACAM_BIN and os.path.exists(LACAM_BIN): !chmod +x {LACAM_BIN}
else: LACAM_BIN=None
RESULTS_DIR='/kaggle/working/results_step2'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Load the v2 model
Note `in_channels=9` here.

In [ ]:
from railgun_plus.models import RailgunUNet
from railgun_plus.data.features_v2 import NUM_FEATURE_CHANNELS_V2

device='cuda' if torch.cuda.is_available() else 'cpu'
model=RailgunUNet(in_channels=NUM_FEATURE_CHANNELS_V2, num_actions=5, base=64).to(device)
ck=torch.load(CKPT, map_location=device)
model.load_state_dict(ck['model']); model.eval()
print('loaded epoch', ck.get('epoch'), '| input channels:', NUM_FEATURE_CHANNELS_V2)

## 3. Build adaptive-density test sets (50 instances at EVERY agent count)

In [ ]:
from railgun_plus.data.test_sets import build_test_sets
AGENT_COUNTS=[16, 32, 48, 64, 80, 96, 128]
test_sets = build_test_sets(
    AGENT_COUNTS, n_per=50, map_size=32,
    save_path=f'{RESULTS_DIR}/test_sets_adaptive.pkl')
print('final:', {k: len(v) for k, v in test_sets.items()})

## 4. Run all methods
Note: the corrector auto-detects from the model that it has 9 input channels and builds v2 features at inference.

In [ ]:
from railgun_plus.eval.harness import run_sweep, sweep_to_table, plot_sweep
sweep = run_sweep(model, test_sets, device=device, lacam_bin=LACAM_BIN)

## 5. Results table

In [ ]:
import pandas as pd
rows = sweep_to_table(sweep, out_csv=f'{RESULTS_DIR}/step2_table.csv')
df = pd.DataFrame(rows)
print('=== CSR ==='); display(df.pivot(index='agents', columns='method', values='csr'))
print('=== Deadlock rate ==='); display(df.pivot(index='agents', columns='method', values='deadlock_rate'))
print('=== SoC ratio ==='); display(df.pivot(index='agents', columns='method', values='avg_soc_ratio'))

## 6. Plots

In [ ]:
plot_sweep(sweep, metric='csr', title='Step 2 (v2 features): CSR vs agents', savepath=f'{RESULTS_DIR}/step2_csr.png')

In [ ]:
plot_sweep(sweep, metric='deadlock_rate', title='Step 2: deadlock rate', savepath=f'{RESULTS_DIR}/step2_deadlock.png')

In [ ]:
plot_sweep(sweep, metric='avg_soc_ratio_solved', title='Step 2: SoC ratio', savepath=f'{RESULTS_DIR}/step2_soc.png')

## 7. Step-2 verdict
Same logic as Step 1: how many densities does `corrected` now beat `pibt_only`?

In [ ]:
baseline_pibt = {k: sweep['pibt_only'][k]['csr'] for k in sweep['pibt_only']}
corrected = {k: sweep['corrected'][k]['csr'] for k in sweep['corrected']}
print('pibt_only CSR:', {k: round(v,3) for k,v in baseline_pibt.items()})
print('corrected CSR:', {k: round(v,3) for k,v in corrected.items()})
deltas = {k: round(corrected[k] - baseline_pibt[k], 3) for k in baseline_pibt}
wins = sum(1 for v in deltas.values() if v > 0)
print(f'deltas vs pibt_only: {deltas}')
print(f'corrected beats pibt_only at {wins}/{len(deltas)} densities')

## 8. Update JOURNAL.md
Manual: paste the Step-2 result block into the journal. The next decision (stop vs Step 3) depends on `wins`.